# Auto-Load CSVs to Delta Tables

This notebook reads all CSV files from the Lakehouse **Files/** section and writes them as Delta tables to **Tables/**.

**Automation:** Attach this notebook to a Data Factory Pipeline or set a schedule trigger in Fabric.

In [ ]:
# ── Configuration ──
FILES_ROOT = "Files/"
WRITE_MODE = "overwrite"  # "overwrite" replaces table each run; "append" adds rows

In [ ]:
# ── Discover CSV files ──
from pyspark.sql import SparkSession
from notebookutils import mssparkutils
import os

spark = SparkSession.builder.getOrCreate()

file_infos = mssparkutils.fs.ls(FILES_ROOT)
csv_files = {}
for f in file_infos:
    if f.name.lower().endswith(".csv"):
        table_name = os.path.splitext(f.name)[0]
        csv_files[table_name] = f"{FILES_ROOT}{f.name}"

print(f"Found {len(csv_files)} CSV file(s) to load:")
for table_name, path in csv_files.items():
    print(f"  {path} -> Tables/{table_name}")

In [ ]:
# ── Load each CSV as a Delta table ──
loaded = []
failed = []

for table_name, csv_path in csv_files.items():
    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(csv_path))

        row_count = df.count()

        (df.write
         .format("delta")
         .mode(WRITE_MODE)
         .saveAsTable(table_name))

        print(f"  ✓ {table_name}: {row_count} rows loaded ({WRITE_MODE})")
        loaded.append(table_name)
    except Exception as e:
        print(f"  ✗ {table_name}: {e}")
        failed.append(table_name)

In [ ]:
# ── Summary ──
print(f"\nDone. Loaded: {len(loaded)}, Failed: {len(failed)}")
if failed:
    print(f"Failed tables: {', '.join(failed)}")